# Motorsport Decision Confidence Engine (MDCE) Demo

This notebook demonstrates the same deterministic logic used by the Streamlit app.

MDCE does **not** try to predict a perfect race strategy. It evaluates whether a strategy recommendation should be trusted under uncertain telemetry, model, and race-context conditions.

## What This Notebook Proves

1. Normal data produces a recommendation and confidence score.
2. Missing telemetry lowers confidence.
3. Model mismatch lowers confidence.
4. Tyre signal drift creates a tyre/lap conflict.
5. Safety-car context changes decision trust.
6. Every risk produces reasons and fallback actions.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import pandas as pd
import plotly.graph_objects as go

from src.data_loader import load_sample_race
from src.models import ScenarioFlags
from src.pipeline import analyze_decision


## Load Offline Demo Data

The dataset is intentionally small and demo-safe. It contains real-style racing fields plus clearly synthetic proxy fields such as `tyre_temp_proxy_c`, `predicted_lap_time_s`, and `speed_consistency`.

In [ ]:
records = load_sample_race(ROOT / "data" / "sample_race.csv")
df = pd.DataFrame([record.__dict__ for record in records])
df.head()

In [ ]:
def plot_laps(records, title):
    frame = pd.DataFrame([record.__dict__ for record in records])
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=frame["lap"], y=frame["lap_time_s"], mode="lines+markers", name="Actual lap time"))
    fig.add_trace(go.Scatter(x=frame["lap"], y=frame["predicted_lap_time_s"], mode="lines+markers", name="Predicted lap time"))
    missing = frame[frame["missing"]]
    if not missing.empty:
        fig.add_trace(go.Scatter(x=missing["lap"], y=missing["lap_time_s"], mode="markers", name="Missing telemetry", marker={"size": 12, "symbol": "x"}))
    fig.update_layout(title=title, xaxis_title="Lap", yaxis_title="Lap time (s)", height=420)
    return fig

plot_laps(records, "Baseline race data")

## Helper: Run A Scenario

The scenario runner calls the same `analyze_decision()` pipeline as the app.

In [ ]:
def run_scenario(name, flags):
    result, scenario_records, notes, conflict = analyze_decision(records, flags, prefer_granite=False)
    rows = []
    for issue in result.issues:
        rows.append({
            "issue": issue.issue,
            "severity": issue.severity.value,
            "penalty": issue.penalty,
            "affected_decisions": ", ".join(issue.affected_decisions),
            "reason": issue.reason,
        })
    summary = {
        "scenario": name,
        "recommendation": result.recommendation.recommendation_type.value,
        "recommended_lap": result.recommendation.recommended_lap,
        "confidence": result.confidence.confidence,
        "risk_level": result.confidence.risk_level,
        "conflict_score": conflict[0],
        "conflict_label": conflict[1],
        "notes": " ".join(notes),
        "fallback": " ".join(result.fallback_actions),
    }
    return summary, pd.DataFrame(rows), result.explanation, scenario_records


## Scenario 1: Normal Data

This is the control case. Confidence should stay higher because the model and data are internally consistent enough.

In [ ]:
normal_summary, normal_issues, normal_explanation, normal_records = run_scenario("Normal data", ScenarioFlags())
normal_summary, normal_issues, normal_explanation

## Scenario 2: Missing Telemetry

The last two laps are marked as missing. Confidence should drop because recent trend evidence is incomplete.

In [ ]:
missing_summary, missing_issues, missing_explanation, missing_records = run_scenario(
    "Missing telemetry",
    ScenarioFlags(missing_telemetry=True),
)
missing_summary, missing_issues, missing_explanation

In [ ]:
plot_laps(missing_records, "Missing telemetry scenario")

## Scenario 3: Model Mismatch

The model is made optimistic while actual lap times worsen. MDCE should detect a model-vs-reality conflict.

In [ ]:
mismatch_summary, mismatch_issues, mismatch_explanation, mismatch_records = run_scenario(
    "Model mismatch",
    ScenarioFlags(model_mismatch=True),
)
mismatch_summary, mismatch_issues, mismatch_explanation

In [ ]:
plot_laps(mismatch_records, "Model mismatch scenario")

## Scenario 4: Tyre Signal Drift

The tyre proxy is held flat while lap times degrade. This shows why MDCE should not blindly trust one stable-looking signal.

In [ ]:
tyre_summary, tyre_issues, tyre_explanation, tyre_records = run_scenario(
    "Tyre signal drift",
    ScenarioFlags(tyre_signal_drift=True),
)
tyre_summary, tyre_issues, tyre_explanation

## Scenario 5: Safety Car Context

Recent laps are marked as Safety Car laps. The recommendation can still exist, but confidence should be reduced because normal degradation assumptions are weaker.

In [ ]:
sc_summary, sc_issues, sc_explanation, sc_records = run_scenario(
    "Safety car phase",
    ScenarioFlags(safety_car_phase=True),
)
sc_summary, sc_issues, sc_explanation

## Scenario Comparison

This table is the simplest proof that MDCE is a trust layer: the recommendation can remain visible while confidence changes based on evidence quality.

In [ ]:
comparison = pd.DataFrame([
    normal_summary,
    missing_summary,
    mismatch_summary,
    tyre_summary,
    sc_summary,
])
comparison[["scenario", "recommendation", "recommended_lap", "confidence", "risk_level", "conflict_score", "conflict_label", "fallback"]]

## Final Takeaway

The project should be presented as:

> MDCE estimates when a race strategy recommendation should or should not be trusted under uncertainty.

It should **not** be presented as a perfect strategy optimizer or a replacement for race engineers.